In [28]:
import os
import os.path as op
import glob 

import pandas as pd
import numpy as np
import xarray as xr

import matplotlib.pyplot as plt

from bluemath_tk.core.io import load_model
from bluemath_tk.core.operations import spatial_gradient

from utils.preprocess_gcms import process_ds_mslp_mask

from functools import partial


#### User Inputs

In [29]:
model_import = 'IPSL-CM6A-LR' #'MIROC6' , 'EC-Earth3-Veg'
model_export = 'ipsl_cm6a_lr' #'miroc6' , 'ec_earth3_veg_lr'
ssp_import = 'ssp2' # ssp2, ssp5
ssp_export = 'ssp245' # ssp245, ssp585

#### Paths

In [30]:
#### paths
path_out = os.path.join(os.getcwd(), "outputs")
path_database = "/lustre/geocean/DATA/PROJECTIONS/MSLP/CMIP6"

if not op.exists(op.join(os.getcwd(), "outputs", "cmip6_models")):
    os.makedirs(op.join(os.getcwd(), "outputs", "cmip6_models"))

In [31]:
dataset = xr.open_dataset('/lustre/geocean/DATA/PROJECTIONS/MSLP/CMIP6/Projections/ssp2/psl_day_EC-Earth3-Veg_ssp245_r1i1p1f1_gr_20960101-20961231.nc')
lats = dataset.lat.values
lons = dataset.lon.values

def preprocess(ds, lats, lons):

    ds = ds.interp(
        lat=lats,
        lon=lons,
        method="linear",
        kwargs={"fill_value": "extrapolate"}
    ).compute()

    return ds

preprocess_fn = partial(
    preprocess,
    lats=lats,
    lons=lons
)

#### Load ERA5 Data

In [32]:
file_exists = os.path.exists('outputs/era5/mslp_era5_1deg.nc')

if file_exists:
    era5_data_1deg = xr.load_dataset('outputs/era5/mslp_era5_1deg.nc') 
else:
    data_slp = xr.open_dataset('/lustre/geocean/WORK/users/montanoj/personal/NorthCarolina_Emulator/inputs/global_mslp_1day_1degree.nc')
    data_slp["mslp"] = data_slp["mslp"]  # Convert from Pa to hPa
    data_slp["mslp_gradient"] = spatial_gradient(data_slp["mslp"])

    data_slp = data_slp.assign_coords(longitude=((data_slp.longitude + 180) % 360) - 180)
    data_slp = data_slp.sortby("longitude")
    
    data_slp.to_netcdf('outputs/era5/mslp_era5_1deg.nc')

    era5_data_1deg = data_slp

In [33]:
new_lat = era5_data_1deg["latitude"].values
new_lon = era5_data_1deg["longitude"].values

#### Load GMC Data

In [34]:
os.makedirs(f'outputs/cmip6_models/{model_export}/historical',exist_ok=True)

file_exists = os.path.exists(f'outputs/cmip6_models/{model_export}/historical/mslp_{model_export}_historical_1deg.nc')

if file_exists:
    ds_hist_gcm_model_interp = xr.open_dataset(f'outputs/cmip6_models/{model_export}/historical/mslp_{model_export}_historical_1deg.nc')

else:
    path_full_hist = op.join(path_database, "Historical")
    path_hist_gcm_model = glob.glob(op.join(path_full_hist, "*" + model_import + "_*.nc"))

    ds_hist_gcm_model = xr.open_mfdataset(path_hist_gcm_model)[["psl"]].rename({"lon": "longitude", "lat": "latitude", "psl": "mslp"})
    ds_hist_gcm_model["mslp"] = ds_hist_gcm_model["mslp"]
    ds_hist_gcm_model["time"] = ds_hist_gcm_model.time.astype("datetime64[ns]")
    ds_hist_gcm_model = ds_hist_gcm_model.resample(time="1D").mean()
    ds_hist_gcm_model = ds_hist_gcm_model.fillna(1013)

    ds_hist_gcm_model = ds_hist_gcm_model.assign_coords(longitude=((ds_hist_gcm_model.longitude + 180) % 360) - 180)
    ds_hist_gcm_model = ds_hist_gcm_model.sortby("longitude")

    ds_hist_gcm_model = ds_hist_gcm_model.chunk({"time": 100, "latitude": 50, "longitude": 50})

    ds_hist_gcm_model_interp = ds_hist_gcm_model.interp(
        latitude=new_lat,
        longitude=new_lon,
        method="linear"
    ).compute()

    ds_hist_gcm_model_interp["mslp_gradient"] = spatial_gradient(ds_hist_gcm_model_interp["mslp"])

    ds_hist_gcm_model_interp.to_netcdf(f'outputs/cmip6_models/{model_export}/historical/mslp_{model_export}_historical_1deg.nc')

#### Load SSP Data

In [35]:
os.makedirs(f'outputs/cmip6_models/{model_export}/{ssp_export}',exist_ok=True)

file_exists = os.path.exists(f'outputs/cmip6_models/{model_export}/{ssp_export}/mslp_{model_export}_{ssp_export}_1deg.nc')

#if file_exists:
if False:
    ds_proj_gcm_model_interp = xr.open_dataset(f'outputs/cmip6_models/{model_export}/{ssp_export}/mslp_{model_export}_{ssp_export}_1deg.nc')
else:

    path_full_proj = op.join(path_database, "Projections", ssp_import)
    path_proj_gcm_model = glob.glob(op.join(path_full_proj, "*" + model_import + "_*.nc"))

    ds_proj_gcm_model = xr.open_mfdataset(path_proj_gcm_model, preprocess=preprocess_fn)[["psl"]].rename({"lon": "longitude", "lat": "latitude", "psl": "mslp"})
    #ds_proj_gcm_model = xr.open_mfdataset(path_proj_gcm_model)[["psl"]].rename({"lon": "longitude", "lat": "latitude", "psl": "mslp"})    
    ds_proj_gcm_model["mslp"] = ds_proj_gcm_model["mslp"]
    ds_proj_gcm_model["time"] = ds_proj_gcm_model.time.astype("datetime64[ns]")

    ds_proj_gcm_model = ds_proj_gcm_model.resample(time="1D").mean()
    ds_proj_gcm_model = ds_proj_gcm_model.fillna(101300)
    ds_proj_gcm_model = ds_proj_gcm_model.chunk({"time": 100, "latitude": 50, "longitude": 50})

    ds_proj_gcm_model = ds_proj_gcm_model.assign_coords(longitude=((ds_proj_gcm_model.longitude + 180) % 360) - 180)
    ds_proj_gcm_model = ds_proj_gcm_model.sortby("longitude")

    ds_proj_gcm_model_interp = ds_proj_gcm_model.interp(
        latitude=new_lat,
        longitude=new_lon,
        method="linear",
        kwargs={"fill_value": "extrapolate"}
    ).compute()

    ds_proj_gcm_model_interp["mslp_gradient"] = spatial_gradient(ds_proj_gcm_model_interp["mslp"])
    
    ds_proj_gcm_model_interp.to_netcdf(f'outputs/cmip6_models/{model_export}/{ssp_export}/mslp_{model_export}_{ssp_export}_1deg.nc')

In [36]:
start_era5 = era5_data_1deg.time.min().values
end_era5 = era5_data_1deg.time.max().values
start_gcm = ds_hist_gcm_model_interp.time.min().values
end_gcm = ds_hist_gcm_model_interp.time.max().values

start_common = max(start_era5, start_gcm)
end_common = min(end_era5, end_gcm)

#### Bias Correction MSLP

In [37]:
ds_hist_gcm_model_1deg_interp_bias = ds_hist_gcm_model_interp.copy()
ds_proj_gcm_model_1deg_interp_bias = ds_proj_gcm_model_interp.copy()

hist_gcm_model_mean = ds_hist_gcm_model_1deg_interp_bias["mslp"].sel(time=slice(start_common, end_common)).mean(dim="time")
hist_gcm_model_std = ds_hist_gcm_model_1deg_interp_bias["mslp"].sel(time=slice(start_common, end_common)).std(dim="time")

hist_era5_mean = era5_data_1deg["mslp"].sel(time=slice(start_common, end_common)).mean(dim="time")
hist_era5_std = era5_data_1deg["mslp"].sel(time=slice(start_common, end_common)).std(dim="time")

ds_hist_gcm_model_1deg_interp_bias["mslp"] = ((ds_hist_gcm_model_1deg_interp_bias["mslp"] - hist_gcm_model_mean) / hist_gcm_model_std ) * hist_era5_std + hist_era5_mean
ds_proj_gcm_model_1deg_interp_bias["mslp"] = ((ds_proj_gcm_model_1deg_interp_bias["mslp"] - hist_gcm_model_mean) / hist_gcm_model_std ) * hist_era5_std + hist_era5_mean

#### Bias Correction MSLP_gradient

In [38]:
# Hist GCM Mean snd STD
hist_gcm_model_mean = ds_hist_gcm_model_1deg_interp_bias["mslp_gradient"].sel(time=slice(start_common, end_common)).mean(dim="time")
hist_gcm_model_std = ds_hist_gcm_model_1deg_interp_bias["mslp_gradient"].sel(time=slice(start_common, end_common)).std(dim="time")

# Hist ERA5 Mean and STD
hist_era5_mean = era5_data_1deg["mslp_gradient"].sel(time=slice(start_common, end_common)).mean(dim="time")
hist_era5_std = era5_data_1deg["mslp_gradient"].sel(time=slice(start_common, end_common)).std(dim="time")

# Bias Correction
ds_hist_gcm_model_1deg_interp_bias["mslp_gradient"] = ( (ds_hist_gcm_model_1deg_interp_bias["mslp_gradient"] - hist_gcm_model_mean) / hist_gcm_model_std ) * hist_era5_std + hist_era5_mean
ds_proj_gcm_model_1deg_interp_bias["mslp_gradient"] = ( (ds_proj_gcm_model_1deg_interp_bias["mslp_gradient"] - hist_gcm_model_mean) / hist_gcm_model_std ) * hist_era5_std + hist_era5_mean

#### Save Outputs

In [39]:
ds_hist_gcm_model_1deg_interp_bias.to_netcdf(f'outputs/cmip6_models/{model_export}/historical/mslp_{model_export}_historical_1deg_bias_corrected.nc')
ds_proj_gcm_model_1deg_interp_bias.to_netcdf(f'outputs/cmip6_models/{model_export}/{ssp_export}/mslp_{model_export}_{ssp_export}_1deg_bias_corrected.nc')